# Today's Market Overview

Cross-asset market dashboard split into:

1. Equity markets
2. Commodities
3. FX
4. Fixed income
5. Money markets

Includes interactive equity sector/industry treemap and world equity performance map.

In [46]:
from datetime import datetime, timedelta
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', 80)
pd.set_option('display.width', 180)

asof = datetime.now()
print(f"Market overview as of {asof:%Y-%m-%d %H:%M:%S}")

Market overview as of 2026-05-15 12:27:00


In [49]:
# Universe definitions
# Bloomberg tickers are the primary source. Yahoo symbols are only used as a fallback for broad indexes.

assets = [
    # Equity indexes
    {"asset_class":"Equity", "region":"US", "country":"United States", "name":"S&P 500", "bbg":"SPX Index", "yahoo":"^GSPC"},
    {"asset_class":"Equity", "region":"US", "country":"United States", "name":"Nasdaq 100", "bbg":"NDX Index", "yahoo":"^NDX"},
    {"asset_class":"Equity", "region":"US", "country":"United States", "name":"Dow Jones", "bbg":"INDU Index", "yahoo":"^DJI"},
    {"asset_class":"Equity", "region":"US", "country":"United States", "name":"Russell 2000", "bbg":"RTY Index", "yahoo":"^RUT"},
    {"asset_class":"Equity", "region":"Europe", "country":"Euro Area", "name":"Euro Stoxx 50", "bbg":"SX5E Index", "yahoo":"^STOXX50E"},
    {"asset_class":"Equity", "region":"Europe", "country":"United Kingdom", "name":"FTSE 100", "bbg":"UKX Index", "yahoo":"^FTSE"},
    {"asset_class":"Equity", "region":"Europe", "country":"Germany", "name":"DAX", "bbg":"DAX Index", "yahoo":"^GDAXI"},
    {"asset_class":"Equity", "region":"Europe", "country":"France", "name":"CAC 40", "bbg":"CAC Index", "yahoo":"^FCHI"},
    {"asset_class":"Equity", "region":"Asia", "country":"Japan", "name":"Nikkei 225", "bbg":"NKY Index", "yahoo":"^N225"},
    {"asset_class":"Equity", "region":"Asia", "country":"Hong Kong", "name":"Hang Seng", "bbg":"HSI Index", "yahoo":"^HSI"},
    {"asset_class":"Equity", "region":"Asia", "country":"China", "name":"CSI 300", "bbg":"SHSZ300 Index", "yahoo":"000300.SS"},
    {"asset_class":"Equity", "region":"Asia", "country":"India", "name":"Nifty 50", "bbg":"NIFTY Index", "yahoo":"^NSEI"},
    {"asset_class":"Equity", "region":"Asia", "country":"Australia", "name":"ASX 200", "bbg":"AS51 Index", "yahoo":"^AXJO"},
    {"asset_class":"Equity", "region":"Americas", "country":"Canada", "name":"TSX Composite", "bbg":"SPTSX Index", "yahoo":"^GSPTSE"},
    {"asset_class":"Equity", "region":"Americas", "country":"Brazil", "name":"Bovespa", "bbg":"IBOV Index", "yahoo":"^BVSP"},
    {"asset_class":"Equity", "region":"Americas", "country":"Mexico", "name":"IPC Mexico", "bbg":"MEXBOL Index", "yahoo":"^MXX"},

    # Commodities
    {"asset_class":"Commodity", "region":"Global", "country":"Global", "name":"WTI Crude", "bbg":"CL1 Comdty", "yahoo":"CL=F"},
    {"asset_class":"Commodity", "region":"Global", "country":"Global", "name":"Brent Crude", "bbg":"CO1 Comdty", "yahoo":"BZ=F"},
    {"asset_class":"Commodity", "region":"Global", "country":"Global", "name":"Gold", "bbg":"XAU Curncy", "yahoo":"GC=F"},
    {"asset_class":"Commodity", "region":"Global", "country":"Global", "name":"Silver", "bbg":"XAG Curncy", "yahoo":"SI=F"},
    {"asset_class":"Commodity", "region":"Global", "country":"Global", "name":"Copper", "bbg":"HG1 Comdty", "yahoo":"HG=F"},

    # FX
    {"asset_class":"FX", "region":"Global", "country":"Global", "name":"Dollar Index", "bbg":"DXY Curncy", "yahoo":"DX-Y.NYB"},
    {"asset_class":"FX", "region":"Global", "country":"Euro Area", "name":"EUR/USD", "bbg":"EURUSD Curncy", "yahoo":"EURUSD=X"},
    {"asset_class":"FX", "region":"Global", "country":"Japan", "name":"USD/JPY", "bbg":"USDJPY Curncy", "yahoo":"JPY=X"},
    {"asset_class":"FX", "region":"Global", "country":"United Kingdom", "name":"GBP/USD", "bbg":"GBPUSD Curncy", "yahoo":"GBPUSD=X"},
    {"asset_class":"FX", "region":"Global", "country":"Switzerland", "name":"USD/CHF", "bbg":"USDCHF Curncy", "yahoo":"CHF=X"},
    {"asset_class":"FX", "region":"Global", "country":"Canada", "name":"USD/CAD", "bbg":"USDCAD Curncy", "yahoo":"CAD=X"},

    # Fixed income
    {"asset_class":"Fixed Income", "region":"US", "country":"United States", "name":"US 2Y Yield", "bbg":"USGG2YR Index", "yahoo":"^IRX"},
    {"asset_class":"Fixed Income", "region":"US", "country":"United States", "name":"US 10Y Yield", "bbg":"USGG10YR Index", "yahoo":"^TNX"},
    {"asset_class":"Fixed Income", "region":"Europe", "country":"Germany", "name":"Germany 10Y Yield", "bbg":"GDBR10 Index", "yahoo":None},
    {"asset_class":"Fixed Income", "region":"Europe", "country":"United Kingdom", "name":"UK 10Y Yield", "bbg":"GUKG10 Index", "yahoo":None},
    {"asset_class":"Fixed Income", "region":"Asia", "country":"Japan", "name":"Japan 10Y Yield", "bbg":"GJGB10 Index", "yahoo":None},
    {"asset_class":"Fixed Income", "region":"Credit", "country":"United States", "name":"US IG CDX Spread", "bbg":"CDX IG CDSI S43 5Y Corp", "yahoo":None},
    {"asset_class":"Fixed Income", "region":"Credit", "country":"United States", "name":"US HY CDX Spread", "bbg":"CDX HY CDSI S43 5Y Corp", "yahoo":None},

    # Money markets
    {"asset_class":"Money Markets", "region":"US", "country":"United States", "name":"SOFR", "bbg":"SOFRRATE Index", "yahoo":None},
    {"asset_class":"Money Markets", "region":"US", "country":"United States", "name":"EFFR", "bbg":"FEDL01 Index", "yahoo":None},
    {"asset_class":"Money Markets", "region":"Europe", "country":"Euro Area", "name":"€STR", "bbg":"ESTRON Index", "yahoo":None},
    {"asset_class":"Money Markets", "region":"United Kingdom", "country":"United Kingdom", "name":"SONIA", "bbg":"SONIO/N Index", "yahoo":None},
]

asset_map = pd.DataFrame(assets)
asset_map.head()

,asset_class,region,country,name,bbg,yahoo
0,Equity,US,United States,S&P 500,SPX Index,^GSPC
1,Equity,US,United States,Nasdaq 100,NDX Index,^NDX
2,Equity,US,United States,Dow Jones,INDU Index,^DJI
3,Equity,US,United States,Russell 2000,RTY Index,^RUT
4,Equity,Europe,Euro Area,Euro Stoxx 50,SX5E Index,^STOXX50E


In [50]:
def try_bloomberg_snapshot(asset_map):
    """Fetch current Bloomberg reference data. Returns (df, error_message)."""
    try:
        import blpapi
    except Exception as e:
        return None, f"blpapi import failed: {e}"

    session = None
    try:
        sessionOptions = blpapi.SessionOptions()
        sessionOptions.setServerHost("localhost")
        sessionOptions.setServerPort(8194)
        session = blpapi.Session(sessionOptions)
        if not session.start():
            raise RuntimeError("Failed to start Bloomberg session")
        if not session.openService("//blp/refdata"):
            raise RuntimeError("Failed to open //blp/refdata")

        refDataService = session.getService("//blp/refdata")
        request = refDataService.createRequest("ReferenceDataRequest")
        for ticker in asset_map['bbg'].dropna().unique():
            request.getElement("securities").appendValue(ticker)
        fields = ["PX_LAST", "CHG_PCT_1D", "NET_CHG", "YLD_YTM_MID", "LAST_UPDATE_DT"]
        for field in fields:
            request.getElement("fields").appendValue(field)
        session.sendRequest(request)

        rows = []
        while True:
            event = session.nextEvent(1000)
            if event.eventType() in (blpapi.Event.PARTIAL_RESPONSE, blpapi.Event.RESPONSE):
                for msg in event:
                    if not msg.hasElement("securityData"):
                        continue
                    securityDataArray = msg.getElement("securityData")
                    for i in range(securityDataArray.numValues()):
                        securityData = securityDataArray.getValueAsElement(i)
                        ticker = securityData.getElementAsString("security")
                        row = {"bbg": ticker}
                        if securityData.hasElement("securityError"):
                            row["error"] = str(securityData.getElement("securityError"))
                        fieldData = securityData.getElement("fieldData")
                        for field in fields:
                            row[field] = fieldData.getElementAsString(field) if fieldData.hasElement(field) else None
                        rows.append(row)
            if event.eventType() == blpapi.Event.RESPONSE:
                break
        df = pd.DataFrame(rows).merge(asset_map, on="bbg", how="left")
        return df, None
    except Exception as e:
        return None, str(e)
    finally:
        if session is not None:
            try:
                session.stop()
            except Exception:
                pass

bbg_snapshot, bbg_error = try_bloomberg_snapshot(asset_map)
print("Bloomberg error:" if bbg_error else "Bloomberg data loaded successfully", bbg_error or "")
if bbg_snapshot is not None:
    display(bbg_snapshot)

Bloomberg data loaded successfully 


,bbg,PX_LAST,CHG_PCT_1D,NET_CHG,YLD_YTM_MID,LAST_UPDATE_DT,asset_class,region,country,name,yahoo
0,SPX Index,7418.370000,-1.104751,None,None,2026-05-15,Equity,US,United States,S&P 500,^GSPC
1,NDX Index,29221.880000,-1.211685,None,None,2026-05-15,Equity,US,United States,Nasdaq 100,^NDX
2,INDU Index,49600.030000,-0.925685,None,None,2026-05-15,Equity,US,United States,Dow Jones,^DJI
3,RTY Index,2794.063000,-2.410790,None,None,2026-05-15,Equity,US,United States,Russell 2000,^RUT
4,SX5E Index,5827.760000,-1.806246,None,None,2026-05-15,Equity,Europe,Euro Area,Euro Stoxx 50,^STOXX50E
5,UKX Index,10195.370000,-1.711763,None,None,2026-05-15,Equity,Europe,United Kingdom,FTSE 100,^FTSE
6,DAX Index,23950.570000,-2.067732,None,None,2026-05-15,Equity,Europe,Germany,DAX,^GDAXI
7,CAC Index,7952.550000,-1.604995,None,None,2026-05-15,Equity,Europe,France,CAC 40,^FCHI
8,NKY Index,61409.290000,-1.986719,None,None,2026-05-15,Equity,Asia,Japan,Nikkei 225,^N225
9,HSI Index,25962.730000,-1.615481,None,None,2026-05-15,Equity,Asia,Hong Kong,Hang Seng,^HSI


In [51]:
def yahoo_snapshot(asset_map):
    """Fallback using yfinance recent daily prices."""
    try:
        import yfinance as yf
    except ImportError:
        import sys, subprocess
        subprocess.check_call([sys.executable, "-m", "pip", "install", "yfinance", "--quiet"])
        import yfinance as yf

    rows = []
    for _, asset in asset_map.dropna(subset=['yahoo']).iterrows():
        symbol = asset['yahoo']
        try:
            hist = yf.Ticker(symbol).history(period="5d", interval="1d", auto_adjust=False)
            if hist.empty:
                rows.append({**asset.to_dict(), "last": np.nan, "chg_pct_1d": np.nan, "note": "no data"})
                continue
            last = hist['Close'].iloc[-1]
            prev = hist['Close'].iloc[-2] if len(hist) > 1 else np.nan
            chg_pct = (last / prev - 1) * 100 if pd.notna(prev) and prev != 0 else np.nan
            rows.append({**asset.to_dict(), "last": last, "chg_pct_1d": chg_pct, "date": hist.index[-1].date(), "note": "Yahoo fallback"})
        except Exception as e:
            rows.append({**asset.to_dict(), "last": np.nan, "chg_pct_1d": np.nan, "note": f"error: {e}"})
    return pd.DataFrame(rows)

if bbg_snapshot is not None and bbg_error is None:
    snapshot = bbg_snapshot.copy()
    snapshot['last'] = pd.to_numeric(snapshot['PX_LAST'], errors='coerce')
    snapshot['chg_pct_1d'] = pd.to_numeric(snapshot['CHG_PCT_1D'], errors='coerce')
    snapshot['source'] = 'Bloomberg'
else:
    snapshot = yahoo_snapshot(asset_map)
    snapshot['source'] = 'Yahoo Finance fallback'

cols = ['asset_class','region','country','name','last','chg_pct_1d','source']
snapshot_view = snapshot[cols].sort_values(['asset_class','region','name']).reset_index(drop=True)
snapshot_view

,asset_class,region,country,name,last,chg_pct_1d,source
0,Commodity,Global,Global,Brent Crude,109.40000,3.480893,Bloomberg
1,Commodity,Global,Global,Copper,624.75000,-4.872478,Bloomberg
2,Commodity,Global,Global,Gold,4539.40000,-2.420000,Bloomberg
3,Commodity,Global,Global,Silver,76.29400,-8.658200,Bloomberg
4,Commodity,Global,Global,WTI Crude,105.21000,3.993279,Bloomberg
5,Equity,Americas,Brazil,Bovespa,176442.16000,-1.078514,Bloomberg
6,Equity,Americas,Mexico,IPC Mexico,67762.49000,-2.087019,Bloomberg
7,Equity,Americas,Canada,TSX Composite,33723.19000,-1.590626,Bloomberg
8,Equity,Asia,Australia,ASX 200,8630.84300,-0.114238,Bloomberg
9,Equity,Asia,China,CSI 300,4859.59000,-1.119197,Bloomberg


In [52]:
# Executive summary
summary_lines = []

for asset_class in ['Equity', 'Commodity', 'FX', 'Fixed Income', 'Money Markets']:
    group = snapshot_view[snapshot_view['asset_class'].eq(asset_class)]
    valid = group.dropna(subset=['chg_pct_1d'])
    if valid.empty:
        summary_lines.append(f"**{asset_class}:** one-day performance data unavailable for this section.")
        continue
    leaders = valid.sort_values('chg_pct_1d', ascending=False).head(2)
    laggards = valid.sort_values('chg_pct_1d').head(2)
    leader_txt = ", ".join([f"{r['name']} {r['chg_pct_1d']:+.2f}%" for _, r in leaders.iterrows()])
    laggard_txt = ", ".join([f"{r['name']} {r['chg_pct_1d']:+.2f}%" for _, r in laggards.iterrows()])
    summary_lines.append(f"**{asset_class}:** strongest: {leader_txt}; weakest: {laggard_txt}.")

spx = snapshot_view.loc[snapshot_view['name'].eq('S&P 500'), 'chg_pct_1d']
ndx = snapshot_view.loc[snapshot_view['name'].eq('Nasdaq 100'), 'chg_pct_1d']
ten = snapshot_view.loc[snapshot_view['name'].eq('US 10Y Yield'), 'last']
wti = snapshot_view.loc[snapshot_view['name'].eq('WTI Crude'), 'chg_pct_1d']
gold = snapshot_view.loc[snapshot_view['name'].eq('Gold'), 'chg_pct_1d']

headline = []
if not spx.empty and pd.notna(spx.iloc[0]): headline.append(f"S&P 500 {spx.iloc[0]:+.2f}%")
if not ndx.empty and pd.notna(ndx.iloc[0]): headline.append(f"Nasdaq 100 {ndx.iloc[0]:+.2f}%")
if not ten.empty and pd.notna(ten.iloc[0]): headline.append(f"US 10Y yield {ten.iloc[0]:.2f}")
if not wti.empty and pd.notna(wti.iloc[0]): headline.append(f"WTI {wti.iloc[0]:+.2f}%")
if not gold.empty and pd.notna(gold.iloc[0]): headline.append(f"Gold {gold.iloc[0]:+.2f}%")

print(f"Market snapshot as of {asof:%Y-%m-%d %H:%M}. Source: {snapshot_view['source'].dropna().iloc[0] if len(snapshot_view) else 'n/a'}")
print("Headline: " + "; ".join(headline))
print("\n".join(summary_lines))

Market snapshot as of 2026-05-15 12:27. Source: Bloomberg
Headline: S&P 500 -1.10%; Nasdaq 100 -1.21%; US 10Y yield 4.59; WTI +3.99%; Gold -2.42%
**Equity:** strongest: ASX 200 -0.11%, Nifty 50 -0.19%; weakest: Russell 2000 -2.41%, IPC Mexico -2.09%.
**Commodity:** strongest: WTI Crude +3.99%, Brent Crude +3.48%; weakest: Silver -8.66%, Copper -4.87%.
**FX:** strongest: Dollar Index +0.46%, USD/CAD -0.20%; weakest: GBP/USD -0.60%, USD/CHF -0.39%.
**Fixed Income:** strongest: Germany 10Y Yield +4.08%, UK 10Y Yield +3.56%; weakest: US 2Y Yield +1.65%, US 10Y Yield +2.34%.
**Money Markets:** strongest: €STR +0.10%, SONIA +0.03%; weakest: SOFR -0.84%, EFFR +0.00%.


## Equity markets

Global equity index performance plus sector/industry breadth.

In [53]:
equity_snapshot = snapshot_view[snapshot_view['asset_class'].eq('Equity')].copy()
equity_snapshot.sort_values('chg_pct_1d', ascending=False)

,asset_class,region,country,name,last,chg_pct_1d,source
8,Equity,Asia,Australia,ASX 200,8630.843,-0.114238,Bloomberg
11,Equity,Asia,India,Nifty 50,23643.500,-0.194600,Bloomberg
17,Equity,US,United States,Dow Jones,49600.030,-0.925685,Bloomberg
5,Equity,Americas,Brazil,Bovespa,176442.160,-1.078514,Bloomberg
20,Equity,US,United States,S&P 500,7418.370,-1.104751,Bloomberg
9,Equity,Asia,China,CSI 300,4859.590,-1.119197,Bloomberg
18,Equity,US,United States,Nasdaq 100,29221.880,-1.211685,Bloomberg
7,Equity,Americas,Canada,TSX Composite,33723.190,-1.590626,Bloomberg
13,Equity,Europe,France,CAC 40,7952.550,-1.604995,Bloomberg
10,Equity,Asia,Hong Kong,Hang Seng,25962.730,-1.615481,Bloomberg


In [54]:
# Interactive world equity performance map
# Negative performance = red, flat = white, positive performance = blue.
red_white_blue = [(0.0, 'red'), (0.5, 'white'), (1.0, 'blue')]

world_equity = equity_snapshot.dropna(subset=['country', 'chg_pct_1d']).copy()
world_equity = world_equity[~world_equity['country'].isin(['Global', 'Euro Area'])]

fig_map = px.choropleth(
    world_equity,
    locations='country',
    locationmode='country names',
    color='chg_pct_1d',
    hover_name='name',
    hover_data={'country': True, 'region': True, 'last': ':,.2f', 'chg_pct_1d': ':+.2f'},
    color_continuous_scale=red_white_blue,
    color_continuous_midpoint=0,
    title='Global Equity Market Performance — 1-day % change'
)
fig_map.update_layout(height=650, geo=dict(showframe=False, showcoastlines=True, projection_type='natural earth'))
fig_map.show()

## Commodities

In [57]:
commodity_snapshot = snapshot_view[snapshot_view['asset_class'].eq('Commodity')].copy()
commodity_snapshot.sort_values('chg_pct_1d', ascending=False)

,asset_class,region,country,name,last,chg_pct_1d,source
4,Commodity,Global,Global,WTI Crude,105.210,3.993279,Bloomberg
0,Commodity,Global,Global,Brent Crude,109.400,3.480893,Bloomberg
2,Commodity,Global,Global,Gold,4539.400,-2.420000,Bloomberg
1,Commodity,Global,Global,Copper,624.750,-4.872478,Bloomberg
3,Commodity,Global,Global,Silver,76.294,-8.658200,Bloomberg


## FX

In [58]:
fx_snapshot = snapshot_view[snapshot_view['asset_class'].eq('FX')].copy()
fx_snapshot.sort_values('chg_pct_1d', ascending=False)

,asset_class,region,country,name,last,chg_pct_1d,source
21,FX,Global,Global,Dollar Index,99.2700,0.45638,Bloomberg
24,FX,Global,Canada,USD/CAD,1.3747,-0.19640,Bloomberg
26,FX,Global,Japan,USD/JPY,158.7000,-0.20790,Bloomberg
22,FX,Global,Euro Area,EUR/USD,1.1626,-0.36850,Bloomberg
25,FX,Global,Switzerland,USD/CHF,0.7868,-0.39400,Bloomberg
23,FX,Global,United Kingdom,GBP/USD,1.3323,-0.59690,Bloomberg


## Fixed income

In [59]:
fixed_income_snapshot = snapshot_view[snapshot_view['asset_class'].eq('Fixed Income')].copy()
fixed_income_snapshot.sort_values(['region', 'name'])

,asset_class,region,country,name,last,chg_pct_1d,source
27,Fixed Income,Asia,Japan,Japan 10Y Yield,2.71700,3.1510,Bloomberg
28,Fixed Income,Credit,United States,US HY CDX Spread,107.19010,NaN,Bloomberg
29,Fixed Income,Credit,United States,US IG CDX Spread,39.30901,NaN,Bloomberg
30,Fixed Income,Europe,Germany,Germany 10Y Yield,3.16700,4.0750,Bloomberg
31,Fixed Income,Europe,United Kingdom,UK 10Y Yield,5.17200,3.5640,Bloomberg
32,Fixed Income,US,United States,US 10Y Yield,4.58640,2.3385,Bloomberg
33,Fixed Income,US,United States,US 2Y Yield,4.08350,1.6504,Bloomberg


## Money markets

In [60]:
money_markets_snapshot = snapshot_view[snapshot_view['asset_class'].eq('Money Markets')].copy()
money_markets_snapshot.sort_values(['region', 'name'])

,asset_class,region,country,name,last,chg_pct_1d,source
34,Money Markets,Europe,Euro Area,€STR,1.93,0.10370,Bloomberg
35,Money Markets,US,United States,EFFR,3.63,0.00000,Bloomberg
36,Money Markets,US,United States,SOFR,3.56,-0.84000,Bloomberg
37,Money Markets,United Kingdom,United Kingdom,SONIA,3.73,0.02682,Bloomberg


## Company earnings calendars by index

Earnings calendars for major equity indexes using Bloomberg index membership. This follows the same pattern as the archived SPX earnings calendar notebook:

1. Pull index members with `INDX_MEMBERS`.
2. Pull `EXPECTED_REPORT_DT`, `EARN_ANN_DT`, and `SHORT_NAME` for each constituent.
3. Filter to upcoming earnings in the selected window.
4. Display calendars by index and an interactive timeline.

In [61]:
# Earnings calendar configuration
# Add/remove indexes here as needed.
earnings_index_universe = pd.DataFrame([
    {'index_name': 'S&P 500', 'bbg_index': 'SPX Index', 'region': 'US'},
    {'index_name': 'Nasdaq 100', 'bbg_index': 'NDX Index', 'region': 'US'},
    {'index_name': 'Dow Jones', 'bbg_index': 'INDU Index', 'region': 'US'},
    {'index_name': 'Russell 2000', 'bbg_index': 'RTY Index', 'region': 'US'},
    {'index_name': 'Euro Stoxx 50', 'bbg_index': 'SX5E Index', 'region': 'Europe'},
    {'index_name': 'FTSE 100', 'bbg_index': 'UKX Index', 'region': 'Europe'},
    {'index_name': 'DAX', 'bbg_index': 'DAX Index', 'region': 'Europe'},
    {'index_name': 'CAC 40', 'bbg_index': 'CAC Index', 'region': 'Europe'},
    {'index_name': 'Nikkei 225', 'bbg_index': 'NKY Index', 'region': 'Asia'},
    {'index_name': 'Hang Seng', 'bbg_index': 'HSI Index', 'region': 'Asia'},
    {'index_name': 'ASX 200', 'bbg_index': 'AS51 Index', 'region': 'Asia'},
])

calendar_start = pd.Timestamp(datetime.now().date())
calendar_end = calendar_start + pd.Timedelta(days=14)
print(f'Earnings calendar window: {calendar_start:%Y-%m-%d} to {calendar_end:%Y-%m-%d}')
earnings_index_universe

Earnings calendar window: 2026-05-15 to 2026-05-29


,index_name,bbg_index,region
0,S&P 500,SPX Index,US
1,Nasdaq 100,NDX Index,US
2,Dow Jones,INDU Index,US
3,Russell 2000,RTY Index,US
4,Euro Stoxx 50,SX5E Index,Europe
5,FTSE 100,UKX Index,Europe
6,DAX,DAX Index,Europe
7,CAC 40,CAC Index,Europe
8,Nikkei 225,NKY Index,Asia
9,Hang Seng,HSI Index,Asia


In [62]:
# Bloomberg/xbbg implementation adapted from Archived/earnings_calendar.ipynb
# Robust to xbbg returning Narwhals/PyArrow-backed dataframes by converting to pandas.

def _to_pandas(df_like):
    """Convert xbbg/narwhals/pandas-like objects to a pandas DataFrame."""
    if df_like is None:
        return pd.DataFrame()
    if isinstance(df_like, pd.DataFrame):
        return df_like
    if hasattr(df_like, 'to_pandas'):
        return df_like.to_pandas()
    if hasattr(df_like, 'to_native'):
        native = df_like.to_native()
        if hasattr(native, 'to_pandas'):
            return native.to_pandas()
        return pd.DataFrame(native)
    return pd.DataFrame(df_like)

async def get_index_earnings_calendar_xbbg(index_name, bbg_index, region, start_date, end_date):
    """Return upcoming earnings calendar for one Bloomberg equity index."""
    from xbbg import blp

    try:
        members_raw = await blp.abds(bbg_index, 'INDX_MEMBERS')
        members = _to_pandas(members_raw)
        ticker_col_candidates = [
            'Member Ticker and Exchange Code',
            'Member Ticker',
            'Index Member',
            'Member Ticker and Exchange Code '
        ]
        ticker_col = next((c for c in ticker_col_candidates if c in members.columns), None)
        if ticker_col is None:
            raise ValueError(f'Could not find member ticker column. Columns: {list(members.columns)}')

        tickers = members[ticker_col].astype(str).str.strip()
        tickers = tickers[tickers.ne('') & tickers.ne('nan')].tolist()
        tickers = [t if t.endswith(' Equity') else f'{t} Equity' for t in tickers]
        if not tickers:
            return pd.DataFrame(), None

        fields = [
            'EXPECTED_REPORT_DT',
            'EARN_ANN_DT',
            'SHORT_NAME',
            'COUNTRY',
            'GICS_SECTOR_NAME',
            'CUR_MKT_CAP',
            'CHG_PCT_1D'
        ]
        data_raw = await blp.abdp(tickers, fields)
        data = _to_pandas(data_raw)
        if data.empty:
            return pd.DataFrame(), None

        df_pivot = data.pivot(index='ticker', columns='field', values='value').reset_index()
        date_col = 'EXPECTED_REPORT_DT' if 'EXPECTED_REPORT_DT' in df_pivot.columns else 'EARN_ANN_DT'
        if date_col not in df_pivot.columns:
            return pd.DataFrame(), None

        df_pivot[date_col] = pd.to_datetime(df_pivot[date_col], errors='coerce')
        for numeric_col in ['CUR_MKT_CAP', 'CHG_PCT_1D']:
            if numeric_col in df_pivot.columns:
                df_pivot[numeric_col] = pd.to_numeric(df_pivot[numeric_col], errors='coerce')

        calendar = df_pivot[
            (df_pivot[date_col] >= pd.Timestamp(start_date)) &
            (df_pivot[date_col] <= pd.Timestamp(end_date))
        ].copy()

        calendar['earnings_date'] = calendar[date_col]
        calendar['date_field_used'] = date_col
        calendar['index_name'] = index_name
        calendar['bbg_index'] = bbg_index
        calendar['region'] = region
        sort_name_col = 'SHORT_NAME' if 'SHORT_NAME' in calendar.columns else 'ticker'
        calendar = calendar.sort_values(['earnings_date', sort_name_col])
        return calendar, date_col

    except Exception as e:
        print(f'{index_name} ({bbg_index}) earnings calendar failed: {e}')
        return pd.DataFrame(), None

async def build_all_index_earnings_calendars(index_universe, start_date, end_date):
    calendars = []
    metadata = []
    for _, idx in index_universe.iterrows():
        print(f"Fetching earnings calendar for {idx['index_name']} ({idx['bbg_index']})...")
        cal, date_col = await get_index_earnings_calendar_xbbg(
            idx['index_name'], idx['bbg_index'], idx['region'], start_date, end_date
        )
        metadata.append({
            'index_name': idx['index_name'],
            'bbg_index': idx['bbg_index'],
            'date_field_used': date_col,
            'companies_with_earnings': len(cal)
        })
        if cal is not None and not cal.empty:
            calendars.append(cal)
    all_calendars = pd.concat(calendars, ignore_index=True) if calendars else pd.DataFrame()
    return all_calendars, pd.DataFrame(metadata)

all_earnings_calendars, earnings_calendar_summary = await build_all_index_earnings_calendars(
    earnings_index_universe, calendar_start, calendar_end
)

earnings_calendar_summary

Fetching earnings calendar for S&P 500 (SPX Index)...


Fetching earnings calendar for Nasdaq 100 (NDX Index)...


Fetching earnings calendar for Dow Jones (INDU Index)...


Fetching earnings calendar for Russell 2000 (RTY Index)...


Fetching earnings calendar for Euro Stoxx 50 (SX5E Index)...


...

Fetching earnings calendar for CAC 40 (CAC Index)...


Fetching earnings calendar for Nikkei 225 (NKY Index)...


Fetching earnings calendar for Hang Seng (HSI Index)...


Fetching earnings calendar for ASX 200 (AS51 Index)...


,index_name,bbg_index,date_field_used,companies_with_earnings
0,S&P 500,SPX Index,EXPECTED_REPORT_DT,29
1,Nasdaq 100,NDX Index,EXPECTED_REPORT_DT,14
2,Dow Jones,INDU Index,EXPECTED_REPORT_DT,4
3,Russell 2000,RTY Index,EXPECTED_REPORT_DT,102
4,Euro Stoxx 50,SX5E Index,EXPECTED_REPORT_DT,0
5,FTSE 100,UKX Index,EXPECTED_REPORT_DT,13
6,DAX,DAX Index,EXPECTED_REPORT_DT,0
7,CAC 40,CAC Index,EXPECTED_REPORT_DT,1
8,Nikkei 225,NKY Index,EXPECTED_REPORT_DT,24
9,Hang Seng,HSI Index,EXPECTED_REPORT_DT,13


In [63]:
# Consolidated earnings calendar across all selected indexes
if all_earnings_calendars.empty:
    print('No upcoming earnings found in the selected window.')
else:
    earnings_display = all_earnings_calendars.copy()
    for col in ['SHORT_NAME', 'COUNTRY', 'GICS_SECTOR_NAME', 'CUR_MKT_CAP', 'CHG_PCT_1D']:
        if col not in earnings_display.columns:
            earnings_display[col] = np.nan

    earnings_display = earnings_display[[
        'earnings_date', 'region', 'index_name', 'ticker', 'SHORT_NAME', 'COUNTRY',
        'GICS_SECTOR_NAME', 'CUR_MKT_CAP', 'CHG_PCT_1D', 'date_field_used'
    ]].rename(columns={
        'SHORT_NAME': 'company',
        'COUNTRY': 'country',
        'GICS_SECTOR_NAME': 'sector',
        'CUR_MKT_CAP': 'market_cap',
        'CHG_PCT_1D': 'price_chg_pct_1d'
    })

    earnings_display['market_cap'] = pd.to_numeric(earnings_display['market_cap'], errors='coerce')
    earnings_display['price_chg_pct_1d'] = pd.to_numeric(earnings_display['price_chg_pct_1d'], errors='coerce')
    earnings_display = earnings_display.sort_values(['earnings_date', 'index_name', 'company'])

    with pd.option_context('display.max_rows', None):
        display(earnings_display)

field,earnings_date,region,index_name,ticker,company,country,sector,market_cap,price_chg_pct_1d,date_field_used
187,2026-05-15,Asia,Hang Seng,3690 HK Equity,MEITUAN-W,CH,Consumer Discretionary,5.106410e+11,-3.500583,EXPECTED_REPORT_DT
163,2026-05-15,Asia,Nikkei 225,2502 JT Equity,ASAHI GROUP HOLD,JN,Consumer Staples,2.336271e+12,0.589391,EXPECTED_REPORT_DT
164,2026-05-15,Asia,Nikkei 225,8331 JT Equity,CHIBA BANK LTD,JN,Financials,1.768188e+12,1.536406,EXPECTED_REPORT_DT
165,2026-05-15,Asia,Nikkei 225,8253 JT Equity,CREDIT SAISON CO,JN,Financials,8.003796e+11,-1.931379,EXPECTED_REPORT_DT
166,2026-05-15,Asia,Nikkei 225,8750 JT Equity,DAIICHI LIFE GRO,JN,Financials,5.860226e+12,7.866667,EXPECTED_REPORT_DT
167,2026-05-15,Asia,Nikkei 225,4324 JT Equity,DENTSU GROUP INC,JN,Communication Services,8.067030e+11,1.555965,EXPECTED_REPORT_DT
168,2026-05-15,Asia,Nikkei 225,6361 JT Equity,EBARA CORP,JN,Industrials,2.530034e+12,-3.421190,EXPECTED_REPORT_DT
169,2026-05-15,Asia,Nikkei 225,4523 JT Equity,EISAI CO LTD,JN,Health Care,1.377459e+12,0.042364,EXPECTED_REPORT_DT
170,2026-05-15,Asia,Nikkei 225,1808 JT Equity,HASEKO,JN,Consumer Discretionary,7.850160e+11,-0.185943,EXPECTED_REPORT_DT
171,2026-05-15,Asia,Nikkei 225,6178 JT Equity,JAPAN POST HOLDI,JN,Financials,5.495637e+12,0.307535,EXPECTED_REPORT_DT


In [64]:
# Interactive earnings calendar table with filters and market-cap sorting

if all_earnings_calendars.empty:
    print('No upcoming earnings to show.')
else:
    import ipywidgets as widgets
    from IPython.display import display, HTML, clear_output

    interactive_df = earnings_display.copy()
    interactive_df['earnings_date'] = pd.to_datetime(interactive_df['earnings_date'])
    interactive_df['market_cap'] = pd.to_numeric(interactive_df['market_cap'], errors='coerce')
    interactive_df['price_chg_pct_1d'] = pd.to_numeric(interactive_df['price_chg_pct_1d'], errors='coerce')

    # Filter controls
    region_options = ['All'] + sorted(interactive_df['region'].dropna().unique().tolist())
    region_dd = widgets.Dropdown(options=region_options, value='All', description='Region:', layout=widgets.Layout(width='220px'))

    index_dd = widgets.Dropdown(options=['All'] + sorted(interactive_df['index_name'].dropna().unique().tolist()), value='All', description='Index:', layout=widgets.Layout(width='260px'))
    sector_dd = widgets.Dropdown(options=['All'] + sorted(interactive_df['sector'].dropna().unique().tolist()), value='All', description='Sector:', layout=widgets.Layout(width='300px'))
    search_box = widgets.Text(value='', placeholder='Search ticker/company', description='Search:', layout=widgets.Layout(width='320px'))

    sort_dd = widgets.Dropdown(
        options=[
            ('Market cap: high to low', 'market_cap_desc'),
            ('Market cap: low to high', 'market_cap_asc'),
            ('Earnings date: soonest first', 'date_asc'),
            ('1-day price move: high to low', 'price_desc'),
            ('Company: A to Z', 'company_asc')
        ],
        value='market_cap_desc',
        description='Sort:',
        layout=widgets.Layout(width='300px')
    )

    output = widgets.Output()

    def _format_market_cap(x):
        if pd.isna(x):
            return ''
        # Bloomberg CUR_MKT_CAP is typically in millions for equities.
        if abs(x) >= 1_000_000:
            return f'{x/1_000_000:,.2f}T'
        if abs(x) >= 1_000:
            return f'{x/1_000:,.2f}B'
        return f'{x:,.0f}M'

    def _render_table(*args):
        df = interactive_df.copy()

        if region_dd.value != 'All':
            df = df[df['region'].eq(region_dd.value)]
        if index_dd.value != 'All':
            df = df[df['index_name'].eq(index_dd.value)]
        if sector_dd.value != 'All':
            df = df[df['sector'].eq(sector_dd.value)]
        if search_box.value.strip():
            q = search_box.value.strip().lower()
            df = df[
                df['ticker'].astype(str).str.lower().str.contains(q, na=False) |
                df['company'].astype(str).str.lower().str.contains(q, na=False)
            ]

        if sort_dd.value == 'market_cap_desc':
            df = df.sort_values('market_cap', ascending=False, na_position='last')
        elif sort_dd.value == 'market_cap_asc':
            df = df.sort_values('market_cap', ascending=True, na_position='last')
        elif sort_dd.value == 'date_asc':
            df = df.sort_values(['earnings_date', 'market_cap'], ascending=[True, False], na_position='last')
        elif sort_dd.value == 'price_desc':
            df = df.sort_values('price_chg_pct_1d', ascending=False, na_position='last')
        elif sort_dd.value == 'company_asc':
            df = df.sort_values('company', ascending=True, na_position='last')

        display_df = df[[
            'earnings_date', 'region', 'index_name', 'ticker', 'company', 'country',
            'sector', 'market_cap', 'price_chg_pct_1d'
        ]].copy()
        display_df['earnings_date'] = display_df['earnings_date'].dt.strftime('%Y-%m-%d')
        display_df['market_cap'] = display_df['market_cap'].map(_format_market_cap)
        display_df['price_chg_pct_1d'] = display_df['price_chg_pct_1d'].map(lambda x: '' if pd.isna(x) else f'{x:+.2f}%')
        display_df = display_df.rename(columns={
            'earnings_date': 'Earnings Date',
            'region': 'Region',
            'index_name': 'Index',
            'ticker': 'Ticker',
            'company': 'Company',
            'country': 'Country',
            'sector': 'Sector',
            'market_cap': 'Market Cap',
            'price_chg_pct_1d': '1D %'
        })

        with output:
            clear_output(wait=True)
            if display_df.empty:
                display(HTML('<b>No companies match the selected filters.</b>'))
                return
            html = f'''
            <style>
            .interactive-earnings-table {{
                border-collapse: collapse;
                width: 100%;
                font-size: 13px;
            }}
            .interactive-earnings-table th {{
                background-color: #1f4e79;
                color: white;
                padding: 7px;
                text-align: left;
                position: sticky;
                top: 0;
                z-index: 2;
            }}
            .interactive-earnings-table td {{
                border-bottom: 1px solid #ddd;
                padding: 5px 7px;
            }}
            .interactive-earnings-table tr:nth-child(even) {{ background-color: #f7f9fc; }}
            .interactive-earnings-table tr:hover {{ background-color: #eef4ff; }}
            .earnings-wrapper {{ max-height: 650px; overflow-y: auto; border: 1px solid #ddd; }}
            </style>
            <h4>Upcoming earnings — {len(display_df)} companies</h4>
            <div class='earnings-wrapper'>
            {display_df.to_html(index=False, escape=False, classes='interactive-earnings-table')}
            </div>
            '''
            display(HTML(html))

    def _refresh_index_options(*args):
        df = interactive_df.copy()
        if region_dd.value != 'All':
            df = df[df['region'].eq(region_dd.value)]
        current = index_dd.value
        options = ['All'] + sorted(df['index_name'].dropna().unique().tolist())
        index_dd.options = options
        index_dd.value = current if current in options else 'All'
        _render_table()

    region_dd.observe(_refresh_index_options, names='value')
    for control in [index_dd, sector_dd, search_box, sort_dd]:
        control.observe(_render_table, names='value')

    controls = widgets.VBox([
        widgets.HBox([region_dd, index_dd, sector_dd]),
        widgets.HBox([search_box, sort_dd])
    ])
    display(controls, output)
    _render_table()

Output()

field,Earnings Date,Region,Index,Ticker,Company,Country,Sector,Market Cap,1D %
,2026-05-15,Asia,Nikkei 225,8306 JT Equity,MITSUBISHI UFJ F,JN,Financials,"34,760,525.28T",+1.81%
,2026-05-15,Asia,Nikkei 225,285A JT Equity,KIOXIA HOLDINGS,JN,Information Technology,"24,273,535.59T",-8.27%
,2026-05-15,Asia,Nikkei 225,8411 JT Equity,MIZUHO FINANCIAL,JN,Financials,"16,884,854.74T",-1.03%
,2026-05-20,Asia,Nikkei 225,8766 JT Equity,TOKIO MARINE HD,JN,Financials,"14,636,512.00T",+2.80%
,2026-05-15,Asia,Nikkei 225,6098 JT Equity,RECRUIT HOLDINGS,JN,Industrials,"11,522,344.97T",+0.58%
,2026-05-20,Asia,Nikkei 225,8725 JT Equity,MS&AD INSURANCE,JN,Financials,"6,553,794.66T",-0.18%
,2026-05-15,Asia,Nikkei 225,8750 JT Equity,DAIICHI LIFE GRO,JN,Financials,"5,860,226.46T",+7.87%
,2026-05-20,Asia,Nikkei 225,8630 JT Equity,SOMPO HOLDINGS I,JN,Financials,"5,698,795.48T",+1.70%
,2026-05-20,US,Nasdaq 100,NVDA UW Equity,NVIDIA CORP,US,Information Technology,"5,506,778.62T",-3.55%
,2026-05-20,US,S&P 500,NVDA UW Equity,NVIDIA CORP,US,Information Technology,"5,506,778.62T",-3.55%


## Enhanced market dashboard

Additional cross-asset panels: volatility, US yield curve, S&P 500 sectors, credit spreads, automated market alerts, and a generated executive summary.

In [67]:
# Derived dashboard panels

def _fmt_signed(x, digits=2):
    if pd.isna(x):
        return "n/a"
    return f"{x:+.{digits}f}"


def _panel_table(panel_name, sort_by="name", ascending=True):
    if enhanced_market_data.empty:
        return pd.DataFrame()
    panel = enhanced_market_data[enhanced_market_data["panel"].eq(panel_name)].copy()
    if panel.empty:
        return panel
    sort_cols = [sort_by] if isinstance(sort_by, str) else sort_by
    return panel.sort_values(sort_cols, ascending=ascending).reset_index(drop=True)

volatility_panel = _panel_table("Volatility", sort_by="name")
yield_curve_panel = _panel_table("US Yield Curve", sort_by="name")
inflation_panel = _panel_table("Inflation", sort_by="name")
credit_panel = _panel_table("Credit", sort_by="name")
sector_panel = _panel_table("S&P 500 Sectors", sort_by="chg_pct_1d", ascending=False)

# For yield series quoted in %, daily_change is in percentage points; convert to bp for display.
if not yield_curve_panel.empty:
    yield_curve_panel["daily_change_bp"] = yield_curve_panel["daily_change"] * 100
if not inflation_panel.empty:
    inflation_panel["daily_change_bp"] = inflation_panel["daily_change"] * 100

# Build simple curve spreads if the inputs are available.
yield_lookup = yield_curve_panel.set_index("name")["last"].to_dict() if not yield_curve_panel.empty else {}
yield_spreads = pd.DataFrame([
    {"spread": "2s10s", "value_bp": (yield_lookup.get("US 10Y") - yield_lookup.get("US 2Y")) * 100 if all(k in yield_lookup for k in ["US 10Y", "US 2Y"]) else np.nan},
    {"spread": "5s30s", "value_bp": (yield_lookup.get("US 30Y") - yield_lookup.get("US 5Y")) * 100 if all(k in yield_lookup for k in ["US 30Y", "US 5Y"]) else np.nan},
    {"spread": "10s30s", "value_bp": (yield_lookup.get("US 30Y") - yield_lookup.get("US 10Y")) * 100 if all(k in yield_lookup for k in ["US 30Y", "US 10Y"]) else np.nan},
]).dropna(subset=["value_bp"])

print("Volatility")
display(volatility_panel[["name", "last", "chg_pct_1d", "daily_change", "last_update_dt"]])

print("US yield curve")
display(yield_curve_panel[["name", "last", "daily_change_bp", "last_update_dt"]])
if not yield_spreads.empty:
    print("Curve spreads")
    display(yield_spreads)

print("Inflation / real rates")
display(inflation_panel[["name", "last", "daily_change_bp", "last_update_dt"]])

print("Credit spreads")
display(credit_panel[["name", "last", "chg_pct_1d", "daily_change", "last_update_dt"]])

print("S&P 500 sectors ranked by 1D performance")
display(sector_panel[["name", "last", "chg_pct_1d", "daily_change", "last_update_dt"]])

Enhanced Bloomberg data loaded successfully 


,panel,name,bbg,unit,last,chg_pct_1d,daily_change,last_update_dt
0,Credit,CDX HY 5Y,CDX HY CDSI S43 5Y Corp,bp,107.19010,NaN,NaN,2026-05-13
1,Credit,CDX IG 5Y,CDX IG CDSI S43 5Y Corp,bp,39.30901,NaN,NaN,2026-05-13
2,Credit,iTraxx Main 5Y,ITRX EUR CDSI S43 5Y Corp,bp,46.55271,4.073334,1.82203,2026-05-15
3,Credit,iTraxx Xover 5Y,ITRX XOVER CDSI S43 5Y Corp,bp,213.86290,3.328439,6.88900,2026-05-15
4,Inflation,US 10Y Breakeven,USGGBE10 Index,%,2.50160,0.490100,0.01220,2026-05-15
5,Inflation,US 10Y Real Yield,USGGT10Y Index,%,2.07480,4.592400,0.09110,2026-05-15
6,S&P 500 Sectors,Communication Services,S5TELS Index,index,503.98000,-0.712573,-3.61700,2026-05-15
7,S&P 500 Sectors,Consumer Discretionary,S5COND Index,index,1939.05600,-1.683252,-33.19800,2026-05-15
8,S&P 500 Sectors,Consumer Staples,S5CONS Index,index,963.67000,-0.112453,-1.08490,2026-05-15
9,S&P 500 Sectors,Energy,S5ENRS Index,index,901.49300,1.655343,14.67980,2026-05-15


### Volatility, sectors, credit, and yield curve panels

In [68]:
# Derived dashboard panels

def _fmt_signed(x, digits=2):
    if pd.isna(x):
        return "n/a"
    return f"{x:+.{digits}f}"


def _panel_table(panel_name, sort_by="name", ascending=True):
    if enhanced_market_data.empty:
        return pd.DataFrame()
    panel = enhanced_market_data[enhanced_market_data["panel"].eq(panel_name)].copy()
    if panel.empty:
        return panel
    sort_cols = [sort_by] if isinstance(sort_by, str) else sort_by
    return panel.sort_values(sort_cols, ascending=ascending).reset_index(drop=True)

volatility_panel = _panel_table("Volatility", sort_by="name")
yield_curve_panel = _panel_table("US Yield Curve", sort_by="name")
inflation_panel = _panel_table("Inflation", sort_by="name")
credit_panel = _panel_table("Credit", sort_by="name")
sector_panel = _panel_table("S&P 500 Sectors", sort_by="chg_pct_1d", ascending=False)

# For yield series quoted in %, daily_change is in percentage points; convert to bp for display.
if not yield_curve_panel.empty:
    yield_curve_panel["daily_change_bp"] = yield_curve_panel["daily_change"] * 100
if not inflation_panel.empty:
    inflation_panel["daily_change_bp"] = inflation_panel["daily_change"] * 100

# Build simple curve spreads if the inputs are available.
yield_lookup = yield_curve_panel.set_index("name")["last"].to_dict() if not yield_curve_panel.empty else {}
yield_spreads = pd.DataFrame([
    {"spread": "2s10s", "value_bp": (yield_lookup.get("US 10Y") - yield_lookup.get("US 2Y")) * 100 if all(k in yield_lookup for k in ["US 10Y", "US 2Y"]) else np.nan},
    {"spread": "5s30s", "value_bp": (yield_lookup.get("US 30Y") - yield_lookup.get("US 5Y")) * 100 if all(k in yield_lookup for k in ["US 30Y", "US 5Y"]) else np.nan},
    {"spread": "10s30s", "value_bp": (yield_lookup.get("US 30Y") - yield_lookup.get("US 10Y")) * 100 if all(k in yield_lookup for k in ["US 30Y", "US 10Y"]) else np.nan},
]).dropna(subset=["value_bp"])

print("Volatility")
display(volatility_panel[["name", "last", "chg_pct_1d", "daily_change", "last_update_dt"]])

print("US yield curve")
display(yield_curve_panel[["name", "last", "daily_change_bp", "last_update_dt"]])
if not yield_spreads.empty:
    print("Curve spreads")
    display(yield_spreads)

print("Inflation / real rates")
display(inflation_panel[["name", "last", "daily_change_bp", "last_update_dt"]])

print("Credit spreads")
display(credit_panel[["name", "last", "chg_pct_1d", "daily_change", "last_update_dt"]])

print("S&P 500 sectors ranked by 1D performance")
display(sector_panel[["name", "last", "chg_pct_1d", "daily_change", "last_update_dt"]])

Volatility


,name,last,chg_pct_1d,daily_change,last_update_dt
0,JPM G7 FX Vol,6.7500,2.580000,0.1700,2026-05-15
1,MOVE,69.6300,-0.868451,-0.6100,2026-05-14
2,VIX,18.1100,4.924681,0.8500,2026-05-15
3,VSTOXX,22.2768,5.212249,1.1036,2026-05-15


US yield curve


,name,last,daily_change_bp,last_update_dt
0,US 10Y,4.5864,10.48,2026-05-15
1,US 2Y,4.0793,6.21,2026-05-15
2,US 30Y,5.1189,9.25,2026-05-15
3,US 5Y,4.2526,10.00,2026-05-15


Curve spreads


...

,name,last,daily_change_bp,last_update_dt
0,US 10Y Breakeven,2.5016,1.22,2026-05-15
1,US 10Y Real Yield,2.0748,9.11,2026-05-15


Credit spreads


,name,last,chg_pct_1d,daily_change,last_update_dt
0,CDX HY 5Y,107.19010,NaN,NaN,2026-05-13
1,CDX IG 5Y,39.30901,NaN,NaN,2026-05-13
2,iTraxx Main 5Y,46.55271,4.073334,1.82203,2026-05-15
3,iTraxx Xover 5Y,213.86290,3.328439,6.88900,2026-05-15


S&P 500 sectors ranked by 1D performance


,name,last,chg_pct_1d,daily_change,last_update_dt
0,Energy,901.493,1.655343,14.6798,2026-05-15
1,Consumer Staples,963.670,-0.112453,-1.0849,2026-05-15
2,Financials,850.460,-0.180892,-1.5412,2026-05-15
3,Communication Services,503.980,-0.712573,-3.6170,2026-05-15
4,Information Technology,6704.220,-0.824649,-55.7460,2026-05-15
5,Health Care,1692.330,-0.878507,-14.9990,2026-05-15
6,Real Estate,275.300,-1.293652,-3.6081,2026-05-15
7,Consumer Discretionary,1939.056,-1.683252,-33.1980,2026-05-15
8,Industrials,1450.380,-1.769647,-26.1290,2026-05-15
9,Utilities,445.560,-2.195251,-10.0007,2026-05-15


### Automated market alerts and executive summary

In [69]:
# Automated alert flags and deterministic executive summary

alert_rows = []

# Alerts from the original broad snapshot.
if "snapshot_view" in globals() and isinstance(snapshot_view, pd.DataFrame) and not snapshot_view.empty:
    broad = snapshot_view.copy()
    broad["chg_pct_1d"] = pd.to_numeric(broad["chg_pct_1d"], errors="coerce")
    for _, row in broad.dropna(subset=["chg_pct_1d"]).iterrows():
        name = row.get("name")
        asset_class = row.get("asset_class")
        move = row.get("chg_pct_1d")
        threshold = 1.5 if asset_class == "Equity" else 2.0 if asset_class == "Commodity" else 0.75 if asset_class == "FX" else None
        if threshold is not None and abs(move) >= threshold:
            alert_rows.append({
                "severity": "High" if abs(move) >= threshold * 1.75 else "Medium",
                "asset": name,
                "category": asset_class,
                "metric": "1D % change",
                "value": move,
                "message": f"{name} moved {_fmt_signed(move)}% today",
            })

# Alerts from enhanced data.
if "enhanced_market_data" in globals() and isinstance(enhanced_market_data, pd.DataFrame) and not enhanced_market_data.empty:
    enhanced = enhanced_market_data.copy()
    enhanced["chg_pct_1d"] = pd.to_numeric(enhanced["chg_pct_1d"], errors="coerce")
    enhanced["daily_change"] = pd.to_numeric(enhanced["daily_change"], errors="coerce")

    for _, row in enhanced[enhanced["panel"].eq("Volatility")].dropna(subset=["chg_pct_1d"]).iterrows():
        if abs(row["chg_pct_1d"]) >= 7.5:
            alert_rows.append({
                "severity": "High" if abs(row["chg_pct_1d"]) >= 15 else "Medium",
                "asset": row["name"],
                "category": "Volatility",
                "metric": "1D % change",
                "value": row["chg_pct_1d"],
                "message": f"{row['name']} volatility index moved {_fmt_signed(row['chg_pct_1d'])}%",
            })

    for _, row in enhanced[enhanced["panel"].isin(["US Yield Curve", "Inflation"])].dropna(subset=["daily_change"]).iterrows():
        move_bp = row["daily_change"] * 100
        threshold_bp = 8 if row["panel"] == "US Yield Curve" else 6
        if abs(move_bp) >= threshold_bp:
            alert_rows.append({
                "severity": "High" if abs(move_bp) >= threshold_bp * 1.75 else "Medium",
                "asset": row["name"],
                "category": "Rates" if row["panel"] == "US Yield Curve" else "Inflation / Real Rates",
                "metric": "1D yield change, bp",
                "value": move_bp,
                "message": f"{row['name']} moved {_fmt_signed(move_bp, 1)} bp",
            })

    for _, row in enhanced[enhanced["panel"].eq("Credit")].iterrows():
        if pd.notna(row.get("daily_change")) and abs(row["daily_change"]) >= 5:
            alert_rows.append({
                "severity": "High" if abs(row["daily_change"]) >= 10 else "Medium",
                "asset": row["name"],
                "category": "Credit",
                "metric": "1D spread change, bp",
                "value": row["daily_change"],
                "message": f"{row['name']} spread moved {_fmt_signed(row['daily_change'], 1)} bp",
            })
        elif pd.notna(row.get("chg_pct_1d")) and abs(row["chg_pct_1d"]) >= 3:
            alert_rows.append({
                "severity": "High" if abs(row["chg_pct_1d"]) >= 6 else "Medium",
                "asset": row["name"],
                "category": "Credit",
                "metric": "1D % change",
                "value": row["chg_pct_1d"],
                "message": f"{row['name']} spread moved {_fmt_signed(row['chg_pct_1d'])}%",
            })

market_alerts = pd.DataFrame(alert_rows)
if not market_alerts.empty:
    severity_order = pd.CategoricalDtype(["High", "Medium", "Low"], ordered=True)
    market_alerts["severity"] = market_alerts["severity"].astype(severity_order)
    market_alerts = market_alerts.sort_values(["severity", "category", "asset"]).reset_index(drop=True)
    display(market_alerts)
else:
    display(pd.DataFrame({"message": ["No major threshold-based market alerts triggered."]}))

# Executive summary helpers.
def _best_worst(df, value_col="chg_pct_1d"):
    valid = df.dropna(subset=[value_col]).copy() if isinstance(df, pd.DataFrame) else pd.DataFrame()
    if valid.empty:
        return None, None
    return valid.loc[valid[value_col].idxmax()], valid.loc[valid[value_col].idxmin()]

summary_lines = []

if "equity_snapshot" in globals() and not equity_snapshot.empty:
    best_eq, worst_eq = _best_worst(equity_snapshot)
    if best_eq is not None:
        summary_lines.append(f"Global equities: best major index is {best_eq['name']} ({_fmt_signed(best_eq['chg_pct_1d'])}%), while {worst_eq['name']} is weakest ({_fmt_signed(worst_eq['chg_pct_1d'])}%).")

if "sector_panel" in globals() and not sector_panel.empty:
    best_sector, worst_sector = _best_worst(sector_panel)
    if best_sector is not None:
        summary_lines.append(f"US sectors: {best_sector['name']} leads ({_fmt_signed(best_sector['chg_pct_1d'])}%), and {worst_sector['name']} lags ({_fmt_signed(worst_sector['chg_pct_1d'])}%).")

if "fx_snapshot" in globals() and not fx_snapshot.empty:
    dxy = fx_snapshot[fx_snapshot["name"].eq("Dollar Index")]
    if not dxy.empty and pd.notna(dxy.iloc[0].get("chg_pct_1d")):
        summary_lines.append(f"FX: the dollar index is {_fmt_signed(dxy.iloc[0]['chg_pct_1d'])}% on the day.")

if "commodity_snapshot" in globals() and not commodity_snapshot.empty:
    best_cmd, worst_cmd = _best_worst(commodity_snapshot)
    if best_cmd is not None:
        summary_lines.append(f"Commodities: {best_cmd['name']} is strongest ({_fmt_signed(best_cmd['chg_pct_1d'])}%), while {worst_cmd['name']} is weakest ({_fmt_signed(worst_cmd['chg_pct_1d'])}%).")

if "yield_curve_panel" in globals() and not yield_curve_panel.empty:
    y10 = yield_curve_panel[yield_curve_panel["name"].eq("US 10Y")]
    if not y10.empty:
        move_bp = y10.iloc[0]["daily_change"] * 100 if pd.notna(y10.iloc[0]["daily_change"]) else np.nan
        summary_lines.append(f"Rates: US 10Y yield is {y10.iloc[0]['last']:.2f}%" + (f", moving {_fmt_signed(move_bp, 1)} bp today." if pd.notna(move_bp) else "."))

if "yield_spreads" in globals() and not yield_spreads.empty:
    curve_bits = ", ".join([f"{r.spread} {r.value_bp:.1f} bp" for r in yield_spreads.itertuples(index=False)])
    summary_lines.append(f"Curve: {curve_bits}.")

if "inflation_panel" in globals() and not inflation_panel.empty:
    real_yield = inflation_panel[inflation_panel["name"].eq("US 10Y Real Yield")]
    breakeven = inflation_panel[inflation_panel["name"].eq("US 10Y Breakeven")]
    bits = []
    if not real_yield.empty:
        bits.append(f"10Y real yield {real_yield.iloc[0]['last']:.2f}%")
    if not breakeven.empty:
        bits.append(f"10Y breakeven {breakeven.iloc[0]['last']:.2f}%")
    if bits:
        summary_lines.append("Inflation/rates: " + ", ".join(bits) + ".")

if "volatility_panel" in globals() and not volatility_panel.empty:
    vix = volatility_panel[volatility_panel["name"].eq("VIX")]
    if not vix.empty and pd.notna(vix.iloc[0].get("last")):
        summary_lines.append(f"Volatility: VIX is {vix.iloc[0]['last']:.1f} ({_fmt_signed(vix.iloc[0].get('chg_pct_1d'))}% today).")

if "credit_panel" in globals() and not credit_panel.empty:
    credit_valid = credit_panel.dropna(subset=["chg_pct_1d"])
    if not credit_valid.empty:
        widest = credit_valid.iloc[credit_valid["chg_pct_1d"].abs().argmax()]
        summary_lines.append(f"Credit: largest listed spread percentage move is {widest['name']} at {_fmt_signed(widest['chg_pct_1d'])}%.")

if not market_alerts.empty:
    high_count = (market_alerts["severity"].astype(str) == "High").sum()
    medium_count = (market_alerts["severity"].astype(str) == "Medium").sum()
    summary_lines.append(f"Alerts: {high_count} high-severity and {medium_count} medium-severity alerts triggered.")
else:
    summary_lines.append("Alerts: no major threshold-based alerts triggered.")

market_executive_summary = "\n".join(f"- {line}" for line in summary_lines)
print(market_executive_summary)

,severity,asset,category,metric,value,message
0,High,Copper,Commodity,1D % change,-4.872478,Copper moved -4.87% today
1,High,Silver,Commodity,1D % change,-8.658200,Silver moved -8.66% today
2,High,WTI Crude,Commodity,1D % change,3.993279,WTI Crude moved +3.99% today
3,Medium,Brent Crude,Commodity,1D % change,3.480893,Brent Crude moved +3.48% today
4,Medium,Gold,Commodity,1D % change,-2.420000,Gold moved -2.42% today
5,Medium,iTraxx Main 5Y,Credit,1D % change,4.073334,iTraxx Main 5Y spread moved +4.07%
6,Medium,iTraxx Xover 5Y,Credit,"1D spread change, bp",6.889000,iTraxx Xover 5Y spread moved +6.9 bp
7,Medium,CAC 40,Equity,1D % change,-1.604995,CAC 40 moved -1.60% today
8,Medium,DAX,Equity,1D % change,-2.067732,DAX moved -2.07% today
9,Medium,Euro Stoxx 50,Equity,1D % change,-1.806246,Euro Stoxx 50 moved -1.81% today


- Global equities: best major index is ASX 200 (-0.11%), while Russell 2000 is weakest (-2.41%).
- US sectors: Energy leads (+1.66%), and Materials lags (-2.50%).
- FX: the dollar index is +0.46% on the day.
- Commodities: WTI Crude is strongest (+3.99%), while Silver is weakest (-8.66%).
- Rates: US 10Y yield is 4.59%, moving +10.5 bp today.
- Curve: 2s10s 50.7 bp, 5s30s 86.6 bp, 10s30s 53.2 bp.
- Inflation/rates: 10Y real yield 2.07%, 10Y breakeven 2.50%.
- Volatility: VIX is 18.1 (+4.92% today).
- Credit: largest listed spread percentage move is iTraxx Main 5Y at +4.07%.
- Alerts: 3 high-severity and 17 medium-severity alerts triggered.
